# VibeScent Week 5 — Qwen3-VL Unified Stack (Kaggle Edition)

**Owner:** Harsh Agarwal  
**Pipeline version:** `w5v1-kaggle`  
**GPU:** P100 (16 GB) or T4 (16 GB) — 4-bit quantization always on

---

## ⚠️ Kaggle vs Colab — Critical Differences

| Topic | Colab | Kaggle | Action |
|-------|-------|--------|--------|
| GPU VRAM | 40–96 GB (A100/Blackwell) | **16 GB** (P100/T4) | 4-bit always |
| bfloat16 | ✅ Native Ampere+ | ❌ P100=sm_6.0, T4=sm_7.5 | **float16** auto-set |
| Flash Attention 2 | ✅ sm_8.0+ | ❌ needs sm_8.0+ | **sdpa/eager** auto-set |
| torch.compile | ✅ worth it on A100 | ⚠️ overhead > gain on P100/T4 | skipped |
| Persistent storage | Google Drive | Kaggle Dataset (output→input) | see Storage guide |
| Secrets | Colab Secrets sidebar | Add-ons → Secrets | HF_TOKEN required |
| Repo source | vibescent.zip from Drive | git clone from GitHub | Internet must be ON |
| Session limit | 12 hr/day | **9–12 hr/session, ~30 hr/wk** | plan ahead |
| Batch rerank _TOPK | 8 | **5** (VRAM constraint) | hardcoded |

---

## Storage Strategy (No Google Drive)

```
/kaggle/working/          ← 20 GB output quota (saved as Kaggle Dataset after run)
  vibescent/              ← cloned repo
  qwen3vl_corpus/         ← corpus embeddings (Stage 0 output, ~540 MB)
  week5/                  ← locked_responses.json, backend_url.txt
  uv_cache/               ← uv package cache (small, persists as output)
/root/.cache/huggingface/ ← HF model cache (~16–20 GB for two 8B models)
                            NOT saved to output (too large for 20 GB quota)
                            Re-downloads each session (~20 min)
/kaggle/input/            ← read-only inputs you've added as datasets
  vibescent-artifacts/    ← optional: your saved output from prior run
    qwen3vl_corpus/       ← skips Stage 0 re-embedding if present
```

### First Run (~35–45 min total):
1. Stage 1: Model downloads from HuggingFace (~20–25 min, two 8B models)
2. Stage 0: Corpus embeddings generated (~10 min, 4-bit T4/P100, batch 32)
3. After all stages pass: Save notebook Output as Kaggle Dataset → name it `vibescent-artifacts`

### Subsequent Runs (~20–25 min total):
1. Add `vibescent-artifacts` as input dataset → Stage 0 skipped (embeddings found)
2. Models still re-download unless you also add an HF cache dataset (advanced, see below)

> **HF Cache advanced tip:** The two 8B models total ~16–20 GB on disk. This exceeds the 20 GB
> output quota so saving hf_cache as output is not practical by default. To eliminate re-downloads:
> create a separate Kaggle Dataset from your hf_cache directory after first run and add it as input
> at path `/kaggle/input/vibescent-hf-cache/`. Stage 1 detects this path and sets `HF_HOME` to it.

---

## Pre-Demo Checklist
- [ ] Notebook settings: **Accelerator = GPU (P100 or T4)**, **Internet = ON**
- [ ] `HF_TOKEN` added via: Add-ons → Secrets → + Add secret (key: `HF_TOKEN`)
- [ ] (Optional) `vibescent-artifacts` dataset added as input → skips Stage 0
- [ ] Run cells top-to-bottom. On fresh session with no artifacts: ~35 min.
- [ ] Copy `CLOUDFLARE_URL` (or `NGROK_URL`) to frontend `.env.local`

---

## Stage Map
| Stage | Description |
|-------|-------------|
| SYNC  | Pull latest code from GitHub — skip on fresh sessions |
| 0     | **ONE-TIME** — Re-embed corpus → `/kaggle/working/qwen3vl_corpus/` |
| 1     | Setup — clone, deps, secrets, GPU detect, ATTN_IMPL |
| 2     | Load Artifacts — corpus embeddings + DataFrame |
| 3     | Load Models — 4-bit, float16, sdpa/eager attention |
| 4     | Build Engine — VibeScentEngine |
| 5     | Start FastAPI |
| 6     | Dual Tunnel — Cloudflare + ngrok |
| 7     | Demo Cache — 5 locked responses |
| 8     | Frontend URL |
| 9     | Smoke Test |
| 10    | Pre-Demo Runbook |

In [ ]:
# SYNC — Force-pull latest code from GitHub
# Run after local git push. Skip on brand-new sessions (Stage 1 clones fresh).
import subprocess, os, sys

REPO_DIR = '/kaggle/working/vibescent'

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('[INFO] Repo not yet cloned — run Stage 1 first.')
else:
    _lfs_env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(['git', '-C', REPO_DIR, 'config', 'filter.lfs.smudge', 'cat'],
                   capture_output=True, env=_lfs_env)
    subprocess.run(['git', '-C', REPO_DIR, 'config', 'filter.lfs.required', 'false'],
                   capture_output=True, env=_lfs_env)

    print('SYNC: Fetching latest from GitHub...')
    r = subprocess.run(
        ['git', '-C', REPO_DIR, 'fetch', 'origin', 'main', '--force'],
        capture_output=True, text=True, env=_lfs_env,
    )
    if r.returncode != 0:
        raise RuntimeError(f'git fetch failed: {r.stderr.strip()}')

    print('SYNC: Resetting to origin/main...')
    r = subprocess.run(
        ['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'],
        capture_output=True, text=True, env=_lfs_env,
    )
    if r.returncode != 0:
        raise RuntimeError(f'git reset failed: {r.stderr.strip()}')

    to_delete = [m for m in sys.modules if m.startswith('vibescents')]
    for m in to_delete:
        del sys.modules[m]
    if to_delete:
        print(f'[OK] Cleared module cache: {", ".join(to_delete)}')

    commit = subprocess.run(
        ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
        capture_output=True, text=True, env=_lfs_env,
    ).stdout.strip()
    print(f'SYNC complete. Commit: {commit}')

In [ ]:
# ============================================================
# STAGE 0 — ONE-TIME corpus re-embedding with Qwen3-VL-Embedding-8B
# Saves /kaggle/working/qwen3vl_corpus/embeddings.npy
# SKIP if /kaggle/input/vibescent-artifacts/qwen3vl_corpus/embeddings.npy exists.
# After completing ALL stages: save notebook Output as Kaggle Dataset
# named 'vibescent-artifacts' and add as input on future runs.
# Runtime: ~10–12 min on T4/P100 with 4-bit, batch 32, ~35k rows.
# ============================================================
import os, sys, subprocess, time, glob, traceback, shutil
import IPython

def custom_exc(shell, etype, evalue, tb, tb_offset=None):
    print('\n' + '='*60)
    print('!!! AN ERROR OCCURRED !!!')
    print('='*60)
    traceback.print_exception(etype, evalue, tb)
    print('='*60 + '\n')

_ipython = IPython.get_ipython()
if _ipython:
    _ipython.set_custom_exc((Exception,), custom_exc)
    _ipython.magic('xmode Verbose')

REPO_DIR     = '/kaggle/working/vibescent'
KAGGLE_WORK  = '/kaggle/working'
GITHUB_REPO  = 'https://github.com/GavinGongTech/VibeScent.git'

# [KAGGLE] HF token via kaggle_secrets (NOT google.colab.userdata)
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN', '')
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded.')
else:
    print('[WARNING] HF_TOKEN not set — add via Add-ons -> Secrets')

# [KAGGLE] HF cache in /root/.cache — NOT counted toward 20 GB output quota
# Check if user has pre-cached models as input dataset (advanced)
_hf_cache_input = '/kaggle/input/vibescent-hf-cache'
HF_CACHE = _hf_cache_input if os.path.isdir(_hf_cache_input) else '/root/.cache/huggingface'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print(f'HF_HOME={HF_CACHE}')

# Clone repo if not present
_lfs_env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
for _cfg in [['filter.lfs.smudge', 'cat'], ['filter.lfs.required', 'false'], ['filter.lfs.process', '']]:
    subprocess.run(['git', 'config', '--global'] + _cfg, env=_lfs_env)

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
    r = subprocess.run(['git', 'clone', '--depth=1', GITHUB_REPO, REPO_DIR],
                       capture_output=True, text=True, env=_lfs_env)
    if r.returncode != 0:
        raise RuntimeError(f'git clone failed: {r.stderr.strip()}')
    print(f'Cloned to {REPO_DIR}')
else:
    print(f'Repo exists at {REPO_DIR}')

os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv', 'hf-transfer'],
               check=True, capture_output=True)
_deps = ['torch>=2.4,<3.0', 'transformers>=4.57,<5.0', 'accelerate>=1.3,<2.0',
         'bitsandbytes', 'qwen-vl-utils>=0.0.14', 'Pillow>=10.0', 'numpy', 'pandas', 'tqdm']
subprocess.run(['uv', 'pip', 'install', '--system', '-q', '-e', REPO_DIR] + _deps,
               check=True, capture_output=True)
print('Deps installed.')

# Output path
OUT_DIR = os.path.join(KAGGLE_WORK, 'qwen3vl_corpus')
os.makedirs(OUT_DIR, exist_ok=True)
OUT_EMB = os.path.join(OUT_DIR, 'embeddings.npy')

# Check for pre-existing embeddings (input dataset or this session)
# [KAGGLE] Intentionally avoid repo artifacts path because it may be a non-npy placeholder.
_EMB_CANDS = [
    '/kaggle/input/vibescent-artifacts/qwen3vl_corpus/embeddings.npy',
    OUT_EMB,
]

import numpy as np
def _try_load_npy(path):
    try:
        arr = np.load(path)
        if not isinstance(arr, np.ndarray) or arr.ndim != 2:
            raise ValueError(f'expected 2D numpy array, got shape={getattr(arr, "shape", None)}')
        return arr
    except Exception as e:
        print(f'[SKIP] Invalid embeddings file: {path} ({e})')
        return None

_existing_path, _existing = None, None
for _p in _EMB_CANDS:
    if os.path.exists(_p):
        _arr = _try_load_npy(_p)
        if _arr is not None:
            _existing_path, _existing = _p, _arr
            break

if _existing_path:
    print(f'[SKIP] Corpus already embedded: {_existing_path}  shape={_existing.shape}')
    if _existing_path != OUT_EMB and not os.path.exists(OUT_EMB):
        shutil.copy(_existing_path, OUT_EMB)
        print(f'Copied to output: {OUT_EMB}')
    print('Stage 0 complete — proceed to Stage 1.')
else:
    import torch, numpy as np, pandas as pd
    from tqdm.auto import tqdm
    from pathlib import Path

    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
    from vibescents.enrich import build_retrieval_text

    _csv_cands = [
        '/kaggle/input/vibescent-artifacts/vibescent_enriched_2k_v2.csv',
        '/kaggle/input/vibescent-artifacts/vibescent_enriched_500_v2.csv',
        os.path.join(REPO_DIR, 'data', 'processed', 'vibescent_unified.csv'),
        os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv'),
    ]
    df = None
    for _p in _csv_cands:
        if os.path.exists(_p):
            df = pd.read_csv(_p, low_memory=False)
            print(f'DataFrame: {_p}  shape={df.shape}')
            break
    if df is None:
        raise FileNotFoundError('No fragrance DataFrame found. Check /kaggle/input/ or repo.')

    df = build_retrieval_text(df)
    texts = df['retrieval_text'].fillna(df.get('name', '')).tolist()
    print(f'Built {len(texts)} retrieval texts. Sample: {texts[0][:80]}')

    from vibescents.embeddings import Qwen3VLMultimodalEmbedder
    from vibescents.settings import Settings

    # [KAGGLE] P100/T4 = 16 GB VRAM → 4-bit always, batch 32 (safe for Stage 0 solo load)
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
    _use_4bit = _vram_gb < 30.0
    Qwen3VLMultimodalEmbedder._BATCH_SIZE = 32 if _use_4bit else 64
    print(f'VRAM={_vram_gb:.1f}GB  4-bit={_use_4bit}  batch={Qwen3VLMultimodalEmbedder._BATCH_SIZE}')

    s = Settings()
    embedder = Qwen3VLMultimodalEmbedder(settings=s, load_in_4bit=_use_4bit)
    print('Model loaded. Starting corpus embedding...')

    CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
    os.makedirs(CKPT_DIR, exist_ok=True)

    def get_checkpoints():
        return sorted(glob.glob(os.path.join(CKPT_DIR, 'embeddings_ckpt_*.npy')),
                      key=lambda p: int(Path(p).stem.split('_')[-1]))

    already_embedded = sum(np.load(f).shape[0] for f in get_checkpoints())
    print(f'Already embedded: {already_embedded} / {len(texts)}')
    texts_to_embed = texts[already_embedded:]

    if texts_to_embed:
        t0 = time.perf_counter()
        CHUNK_SIZE = Qwen3VLMultimodalEmbedder._BATCH_SIZE * 50
        with tqdm(total=len(texts_to_embed), desc='Embedding') as pbar:
            for i in range(0, len(texts_to_embed), CHUNK_SIZE):
                chunk = texts_to_embed[i:i+CHUNK_SIZE]
                chunk_emb = embedder.embed_multimodal_documents(chunk)
                ckpt_path = os.path.join(CKPT_DIR, f'embeddings_ckpt_{len(get_checkpoints())}.npy')
                np.save(ckpt_path, chunk_emb)
                pbar.update(len(chunk))
        print(f'Embedded {len(texts_to_embed)} rows in {time.perf_counter()-t0:.0f}s')

    corpus_emb = np.vstack([np.load(f) for f in get_checkpoints()])
    np.save(OUT_EMB, corpus_emb.astype(np.float32))
    print(f'Saved: {OUT_EMB}')
    print(f'Shape: {corpus_emb.shape}')
    print()
    print('Stage 0 COMPLETE.')
    print('After all stages pass, save notebook Output as Kaggle Dataset: vibescent-artifacts')
    print('Add it as input on future runs to skip Stage 0.')

In [ ]:
# Stage 1: Kaggle Setup
import os, sys, subprocess, time, traceback, shutil
_t0 = time.perf_counter()
PIPELINE_VERSION = 'w5v1-kaggle'
REPO_DIR         = '/kaggle/working/vibescent'
KAGGLE_WORK      = '/kaggle/working'
FASTAPI_PORT     = 8000
FASTAPI_HOST     = '127.0.0.1'
_BASE_URL        = f'http://{FASTAPI_HOST}:{FASTAPI_PORT}'
GITHUB_REPO      = 'https://github.com/GavinGongTech/VibeScent.git'

import IPython

def custom_exc(shell, etype, evalue, tb, tb_offset=None):
    print('\n' + '='*60 + '\n!!! AN ERROR OCCURRED !!!\n' + '='*60)
    traceback.print_exception(etype, evalue, tb)
    print('='*60 + '\n')

_ipython = IPython.get_ipython()
if _ipython:
    _ipython.set_custom_exc((Exception,), custom_exc)
    _ipython.magic('xmode Verbose')

# [KAGGLE] HF token via Kaggle Secrets (NOT google.colab.userdata)
# Add via: Kaggle Notebook -> Add-ons -> Secrets -> + Add secret
# Key: HF_TOKEN   Value: hf_...
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN', '')

if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Kaggle Secrets.')
else:
    print('[WARNING] HF_TOKEN not found.')
    print('          Add via: Add-ons -> Secrets -> HF_TOKEN')
    print('          First-time model download (~20 min) will fail without it.')

# [KAGGLE] HF cache: use pre-cached input dataset if available, else session disk
# /root/.cache/ is NOT counted toward the 20 GB output quota
# To persist: create a Kaggle Dataset from /root/.cache/huggingface after first run
# and add as input at path /kaggle/input/vibescent-hf-cache/
_hf_cache_input = '/kaggle/input/vibescent-hf-cache'
HF_CACHE = _hf_cache_input if os.path.isdir(_hf_cache_input) else '/root/.cache/huggingface'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print(f'HF_HOME={HF_CACHE}')

UV_CACHE = os.path.join(KAGGLE_WORK, 'uv_cache')
os.makedirs(UV_CACHE, exist_ok=True)
os.environ['UV_CACHE_DIR'] = UV_CACHE

# [KAGGLE] Clone from GitHub (no zip upload needed; internet must be ON)
_lfs_env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
for _cfg in [
    ['filter.lfs.smudge', 'cat'],
    ['filter.lfs.required', 'false'],
    ['filter.lfs.process', ''],
]:
    subprocess.run(['git', 'config', '--global'] + _cfg, env=_lfs_env)

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('Syncing existing repo...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'],
                   capture_output=True, env=_lfs_env)
    r = subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'],
                       capture_output=True, text=True, env=_lfs_env)
    if r.returncode != 0:
        raise RuntimeError(f'git reset failed: {r.stderr.strip()}')
else:
    print('Cloning repo...')
    if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
    r = subprocess.run(['git', 'clone', '--depth=1', GITHUB_REPO, REPO_DIR],
                       capture_output=True, text=True, env=_lfs_env)
    if r.returncode != 0:
        raise RuntimeError(f'git clone failed: {r.stderr.strip()}')

os.chdir(REPO_DIR)
commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                        capture_output=True, text=True, env=_lfs_env).stdout.strip()
print(f'Repo: {REPO_DIR}  commit={commit}')

from tqdm.auto import tqdm
bar = tqdm(['Pkg', 'Deps', 'Cloudflared', 'GPU'], desc='Stage 1', ncols=100, unit='step')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv', 'hf-transfer'],
               check=True, capture_output=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-q', '-e', REPO_DIR],
               check=True, capture_output=True)
bar.update(1)

_DEPS = [
    'torch>=2.4,<3.0', 'transformers>=4.57,<5.0', 'accelerate>=1.3,<2.0',
    'bitsandbytes',
    'qwen-vl-utils>=0.0.14', 'fastapi>=0.115,<1.0', 'uvicorn[standard]>=0.34,<1.0',
    'httpx>=0.27,<1.0', 'pyngrok>=7.2,<8.0', 'nest-asyncio>=1.6,<2.0',
    'Pillow>=10.0,<12.0', 'tqdm', 'pandas', 'numpy',
]
subprocess.run(['uv', 'pip', 'install', '--system', '-q'] + _DEPS,
               check=True, capture_output=True)
bar.update(1)

CLOUDFLARED_BIN = '/usr/local/bin/cloudflared'
if not os.path.isfile(CLOUDFLARED_BIN):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', CLOUDFLARED_BIN,
    ], capture_output=True)
    subprocess.run(['chmod', '+x', CLOUDFLARED_BIN], capture_output=True)
bar.update(1)

import torch
GPU_TIER = 'CPU'; DEVICE = 'cpu'; ATTN_IMPL = 'eager'
if torch.cuda.is_available():
    DEVICE = 'cuda'
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    _name = torch.cuda.get_device_name(0)
    _sm   = torch.cuda.get_device_capability(0)

    GPU_TIER = ('P100' if 'P100' in _name else
                ('T4'  if 'T4'  in _name else
                 ('A100' if _vram >= 35 else f'GPU_{_vram:.0f}GB')))

    # [KAGGLE] Attention implementation by compute capability:
    #   sm_6.x (P100): no flash_attention_2, no sdpa optimized path -> 'eager'
    #   sm_7.x (T4):   no flash_attention_2, sdpa works -> 'sdpa'
    #   sm_8.x+ (A100+): flash_attention_2 available (unlikely on Kaggle free tier)
    ATTN_IMPL = ('flash_attention_2' if _sm[0] >= 8 else
                 ('sdpa' if _sm[0] >= 7 else 'eager'))

    print(f'GPU: {_name}')
    print(f'  VRAM={_vram:.1f}GB  sm={_sm[0]}.{_sm[1]}  tier={GPU_TIER}')
    print(f'  ATTN_IMPL={ATTN_IMPL}  (flash_attention_2 needs sm>=8.0)')
    print(f'  dtype=float16  (bfloat16 not natively supported on sm<8.0)')
    print(f'  4-bit=ALWAYS  (16 GB VRAM, two 8B models)')
else:
    print('[WARNING] No GPU detected — inference will be extremely slow.')

bar.update(1); bar.close()
W5_ARTIFACTS = os.path.join(KAGGLE_WORK, 'week5')
os.makedirs(W5_ARTIFACTS, exist_ok=True)
print(f'Setup done in {time.perf_counter()-_t0:.1f}s | GPU={GPU_TIER} | ATTN={ATTN_IMPL}')

In [ ]:
# Stage 2: Load Qwen3VL corpus embeddings + DataFrame
import time, os
import numpy as np
import pandas as pd
_t0 = time.perf_counter()

# [KAGGLE] Search order: input dataset first, then this session's Stage 0 output, then repo
_EMB_CANDS = [
    '/kaggle/input/vibescent-artifacts/qwen3vl_corpus/embeddings.npy',
    os.path.join(KAGGLE_WORK, 'qwen3vl_corpus', 'embeddings.npy'),
    os.path.join(REPO_DIR, 'artifacts', 'qwen3vl_corpus', 'embeddings.npy'),
]
CORPUS_EMB_PATH = next((p for p in _EMB_CANDS if os.path.exists(p)), None)
if CORPUS_EMB_PATH is None:
    raise FileNotFoundError(
        'Qwen3VL corpus embeddings not found.\n'
        'Option A: Run Stage 0 first.\n'
        'Option B: Add vibescent-artifacts Kaggle Dataset as notebook input.\n'
        f'Searched: {_EMB_CANDS}'
    )

print(f'Loading corpus embeddings from {CORPUS_EMB_PATH} ...', end=' ', flush=True)
CORPUS_EMBEDDINGS = np.load(CORPUS_EMB_PATH).astype(np.float32)
_norms = np.linalg.norm(CORPUS_EMBEDDINGS, axis=1, keepdims=True)
CORPUS_EMBEDDINGS = CORPUS_EMBEDDINGS / np.where(_norms < 1e-9, 1.0, _norms)
print(f'done. shape={CORPUS_EMBEDDINGS.shape}')

_CSV_CANDS = [
    '/kaggle/input/vibescent-artifacts/vibescent_enriched_2k_v2.csv',
    '/kaggle/input/vibescent-artifacts/vibescent_enriched_500_v2.csv',
    os.path.join(REPO_DIR, 'data', 'processed', 'vibescent_unified.csv'),
    os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv'),
]
FRAGRANCE_DF = None
for _c in _CSV_CANDS:
    if os.path.exists(_c):
        FRAGRANCE_DF = pd.read_csv(_c, low_memory=False)
        print(f'DataFrame: {_c}  shape={FRAGRANCE_DF.shape}')
        break
if FRAGRANCE_DF is None:
    raise FileNotFoundError('No fragrance DataFrame found.')

if len(CORPUS_EMBEDDINGS) != len(FRAGRANCE_DF):
    raise AssertionError(
        f'Embedding/DataFrame length mismatch: '
        f'emb={len(CORPUS_EMBEDDINGS)} vs df={len(FRAGRANCE_DF)}.\n'
        'Re-run Stage 0 to regenerate corpus embeddings from current DataFrame.'
    )

if 'fragrance_id' not in FRAGRANCE_DF.columns:
    FRAGRANCE_DF.insert(0, 'fragrance_id', FRAGRANCE_DF.index.astype(str))

print(f'\nArtifacts ready in {time.perf_counter()-_t0:.1f}s')
print(f'  Corpus: {CORPUS_EMBEDDINGS.shape}  (Qwen3-VL-Embedding-8B space, L2-normed)')
print(f'  DataFrame: {FRAGRANCE_DF.shape}')

In [ ]:
# Stage 3: Load Qwen3-VL-Embedding-8B + Qwen3-VL-Reranker-8B
#
# [KAGGLE] Differences from Colab:
#   1. 4-bit always required  (P100/T4 = 16 GB VRAM)
#   2. float16, NOT bfloat16  (P100 sm_6.0 / T4 sm_7.5 no native BF16)
#   3. ATTN_IMPL = sdpa/eager (flash_attention_2 requires sm_8.0+)
#   4. torch.compile SKIPPED  (sm_6.x/7.x compile overhead > benefit)
#   5. Embedder batch=4       (conservative; both models loaded simultaneously)
import time, torch, gc, sys, os
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
_t0 = time.perf_counter()

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
_sm      = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)

# [KAGGLE] 4-bit mandatory on 16 GB. Never skip.
_use_4bit = _vram_gb < 30.0

# [KAGGLE] float16: P100/T4 lack native bfloat16 accumulation
_dtype = torch.float16

# ATTN_IMPL carried from Stage 1; safe fallback if cell run standalone
_ATTN = globals().get('ATTN_IMPL', 'sdpa' if _sm[0] >= 7 else 'eager')

print(f'VRAM={_vram_gb:.1f}GB | 4-bit={_use_4bit} | dtype=float16 | attn={_ATTN}')

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── 1. Qwen3-VL-Embedding-8B ─────────────────────────────────────────────
EMBEDDER_MODEL = 'Qwen/Qwen3-VL-Embedding-8B'
print(f'Loading {EMBEDDER_MODEL} ...')
from vibescents.embeddings import Qwen3VLMultimodalEmbedder
from vibescents.settings import Settings

_s = Settings()
# [KAGGLE] batch=4: conservative for 16 GB when reranker also loaded
Qwen3VLMultimodalEmbedder._BATCH_SIZE = 4
VL_EMBEDDER = Qwen3VLMultimodalEmbedder(settings=_s, load_in_4bit=_use_4bit)

_warmup = VL_EMBEDDER.embed_multimodal_documents(['warmup fragrance'])
EMB_DIM = _warmup.shape[1]
_vram_after_emb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f'Embedder ready. dim={EMB_DIM}  VRAM used: {_vram_after_emb:.1f} GB')

assert EMB_DIM == CORPUS_EMBEDDINGS.shape[1], (
    f'Embedding dimension mismatch: model={EMB_DIM} vs corpus={CORPUS_EMBEDDINGS.shape[1]}.\n'
    'Re-run Stage 0 to regenerate corpus embeddings.'
)
print(f'Dimension check passed: {EMB_DIM}')

# ── 2. Qwen3-VL-Reranker-8B ──────────────────────────────────────────────
RERANKER_MODEL = 'Qwen/Qwen3-VL-Reranker-8B'
print(f'\nLoading {RERANKER_MODEL} ...')

from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

VL_RERANKER_PROC = AutoProcessor.from_pretrained(
    RERANKER_MODEL, trust_remote_code=True
)

_reranker_kwargs = {
    'device_map': 'cuda',
    'trust_remote_code': True,
    'attn_implementation': _ATTN,  # [KAGGLE] sdpa or eager, NOT flash_attention_2
}
if _use_4bit:
    _reranker_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True)
else:
    _reranker_kwargs['torch_dtype'] = _dtype  # [KAGGLE] float16

VL_RERANKER_MODEL = AutoModelForCausalLM.from_pretrained(
    RERANKER_MODEL,
    **_reranker_kwargs
).eval()

# [KAGGLE] torch.compile SKIPPED:
#   P100 sm_6.0: Triton backend targets sm_7.0+ for most fusions
#   T4 sm_7.5: compile is valid but overhead > gain at batch=5
print('[KAGGLE] torch.compile skipped (no speedup on P100/T4 for batch<=5 reranking)')

_tok = VL_RERANKER_PROC.tokenizer
_YES_IDS = _tok.encode('yes', add_special_tokens=False)
_NO_IDS  = _tok.encode('no',  add_special_tokens=False)
_YES_ID  = _YES_IDS[0] if _YES_IDS else -1
_NO_ID   = _NO_IDS[0]  if _NO_IDS  else -1

_vram_total = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f'Reranker ready. yes_id={_YES_ID} no_id={_NO_ID}')
print(f'VRAM after both models: {_vram_total:.1f} / {_vram_gb:.0f} GB')

if torch.cuda.is_available() and _vram_total > _vram_gb * 0.82:
    print('[WARNING] VRAM >82% — may OOM during batch reranking.')
    print('          If OOM occurs: reduce _TOPK from 5 to 3 in Stage 4.')

# Move corpus to GPU
print('Moving corpus embeddings to GPU...')
CORPUS_EMBEDDINGS_GPU = torch.from_numpy(CORPUS_EMBEDDINGS).to('cuda', non_blocking=True)
# float32 retained; 35k x 4096 = ~540 MB, fine on 16 GB after 4-bit model loads

print(f'Total VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB / {_vram_gb:.0f} GB')
print(f'\nStage 3 done in {time.perf_counter()-_t0:.0f}s')

In [ ]:
# Stage 4: Build VibeScentEngine — Qwen3-VL unified stack
# [KAGGLE] _TOPK reduced 8->5: smaller batch keeps peak VRAM under 16 GB limit
import base64, io, os, tempfile, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from vibescents.backend_app import create_app
from vibescents.schemas import RecommendRequest, RecommendResponse, FragranceRecommendation
from vibescents.structured_scorer import compute_structured_scores

_TOPK  = 5   # [KAGGLE] 8 on Colab A100; 5 here to stay under 16 GB VRAM
_FINAL = 3

_RERANKER_SYS = (
    'Judge whether the Document meets the requirements based on the Query and '
    'the Instruct provided. Note that the answer can only be yes or no.'
)
_RERANKER_INSTRUCTION = (
    'Given an outfit image and occasion context, assess whether the fragrance '
    'candidate is a strong match. Consider: formality, mood, season, and scent-aesthetic harmony.'
)


def _embed_query(context_str, image_bytes):
    if image_bytes:
        import tempfile as _tf_mod
        with _tf_mod.NamedTemporaryFile(suffix='.jpg', delete=False) as _tf:
            _tf.write(image_bytes)
            _tmp = _tf.name
        try:
            vec = VL_EMBEDDER.embed_multimodal_query(text=context_str, image_path=_tmp)
        finally:
            os.unlink(_tmp)
    else:
        vec = VL_EMBEDDER.embed_multimodal_documents([context_str])
    vec = vec.astype('float32').reshape(1, -1)
    vec /= (np.linalg.norm(vec, axis=1, keepdims=True) + 1e-9)
    return vec


@torch.no_grad()
def _rerank_batch(context_str, frag_texts, image_bytes):
    from qwen_vl_utils import process_vision_info
    import base64 as _b64

    _img_b64 = _b64.b64encode(image_bytes).decode() if image_bytes else None
    texts_for_proc, all_messages = [], []

    for frag_text in frag_texts:
        user_parts = []
        if _img_b64:
            user_parts.append({'type': 'image', 'image': f'data:image/jpeg;base64,{_img_b64}'})
        user_parts.append({
            'type': 'text',
            'text': (
                f'<Instruct>: {_RERANKER_INSTRUCTION}\n'
                f'<Query>: {context_str}\n'
                f'<Document>: {frag_text}'
            ),
        })
        messages = [
            {'role': 'system', 'content': _RERANKER_SYS},
            {'role': 'user',   'content': user_parts},
        ]
        text = VL_RERANKER_PROC.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        text += '<think>\n\n</think>\n\n'
        texts_for_proc.append(text)
        all_messages.append(messages)

    if _img_b64:
        _single_img_msg = [{'role': 'user', 'content': [{'type': 'image', 'image': f'data:image/jpeg;base64,{_img_b64}'}]}]
        _img_inputs_one, _ = process_vision_info(_single_img_msg)
        image_inputs = _img_inputs_one * len(frag_texts)
    else:
        image_inputs = []

    inputs = VL_RERANKER_PROC(
        text=texts_for_proc,
        images=image_inputs if image_inputs else None,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,
    ).to('cuda', non_blocking=True)

    out = VL_RERANKER_MODEL(**inputs)
    last_logits = out.logits[:, -1, :]

    no_l  = last_logits[:, _NO_ID]  if _NO_ID  >= 0 else torch.zeros(last_logits.shape[0], device=last_logits.device)
    yes_l = last_logits[:, _YES_ID] if _YES_ID >= 0 else torch.zeros(last_logits.shape[0], device=last_logits.device)
    p_yes = F.log_softmax(torch.stack([no_l, yes_l], dim=1), dim=1)[:, 1].exp()
    return p_yes.tolist()


def _build_frag_text(row):
    parts = []
    for col in ('name', 'brand', 'top_notes', 'middle_notes', 'base_notes',
                'main_accords', 'vibe_sentence', 'likely_occasion'):
        v = str(row.get(col, '') or '')
        if v and v.lower() not in ('nan', 'none', ''):
            parts.append(v)
    return ' | '.join(parts)[:400]


def _context_str(ctx):
    parts = [p for p in [ctx.eventType, ctx.timeOfDay, ctx.mood,
                         getattr(ctx, 'customNotes', None)] if p]
    return ', '.join(parts) if parts else 'general occasion'


class VibeScentEngine:
    def __init__(self, *, fragrance_df, corpus_embeddings, locked_cache=None):
        self._df     = fragrance_df.reset_index(drop=True)
        self._corpus = corpus_embeddings
        self._cache  = locked_cache or {}

    def recommend(self, *, request, image_bytes):
        t0 = time.perf_counter()
        ctx = request.context
        cache_key = f'{ctx.eventType}|{ctx.timeOfDay}|{ctx.mood}'
        if cache_key in self._cache:
            return RecommendResponse(
                recommendations=[FragranceRecommendation(**r)
                                  for r in self._cache[cache_key]['recommendations']]
            )

        ctx_str = _context_str(ctx)

        q_vec = _embed_query(ctx_str, image_bytes)
        q_gpu = torch.from_numpy(q_vec).to('cuda', non_blocking=True)
        scores = (CORPUS_EMBEDDINGS_GPU @ q_gpu.T).squeeze(1).cpu().numpy()

        struct_scores = compute_structured_scores(ctx, self._df)
        _lo, _hi = struct_scores.min(), struct_scores.max()
        struct_norm = (struct_scores - _lo) / max(_hi - _lo, 1e-9)
        fused = 0.85 * scores + 0.15 * struct_norm

        top_idx = np.argpartition(-fused, _TOPK)[:_TOPK]
        top_idx = top_idx[np.argsort(-fused[top_idx])]

        print(f'  Retrieval ({time.perf_counter()-t0:.1f}s). Top: {self._df.iloc[top_idx[0]].get("name", "?")}')

        frag_texts = [_build_frag_text(self._df.iloc[int(i)]) for i in top_idx]
        try:
            rerank_scores = _rerank_batch(ctx_str, frag_texts, image_bytes)
        except Exception as _e:
            print(f'  Batch reranker failed: {_e} — falling back to fusion scores')
            rerank_scores = [float(fused[int(i)]) for i in top_idx]

        _ranked  = sorted(zip(top_idx, rerank_scores), key=lambda x: x[1], reverse=True)
        final_top = _ranked[:_FINAL]

        print(f'  Reranked ({time.perf_counter()-t0:.1f}s). Final: {self._df.iloc[final_top[0][0]].get("name", "?")}')

        recs = []
        for rank, (idx, rr_score) in enumerate(final_top, start=1):
            row   = self._df.iloc[int(idx)]
            name  = str(row.get('name',  f'Fragrance #{idx}'))
            house = str(row.get('brand', 'Unknown'))
            notes = []
            for col in ('top_notes', 'middle_notes', 'base_notes', 'main_accords'):
                v = str(row.get(col, '') or '')
                if v and v.lower() not in ('nan', 'none', ''):
                    notes += [n.strip() for n in v.split(',') if n.strip()]
            notes = list(dict.fromkeys(notes))[:8]
            vibe = str(row.get('vibe_sentence', '') or '')
            if vibe.lower() in ('nan', 'none', ''): vibe = ''
            reasoning = vibe or f'Matched for {ctx_str} (score: {rr_score:.3f})'
            occ_lbl = str(row.get('likely_occasion', ctx_str) or ctx_str)
            if occ_lbl.lower() in ('nan', 'none', ''): occ_lbl = ctx_str
            recs.append(FragranceRecommendation(
                rank=rank, name=name, house=house,
                score=round(rr_score, 4),
                notes=notes, reasoning=reasoning, occasion=occ_lbl,
            ))

        print(f'  Total latency: {time.perf_counter()-t0:.1f}s')
        return RecommendResponse(recommendations=recs)


ENGINE = VibeScentEngine(
    fragrance_df=FRAGRANCE_DF,
    corpus_embeddings=CORPUS_EMBEDDINGS,
)
print(f'VibeScentEngine (w5v1-kaggle) built. corpus={len(FRAGRANCE_DF)} fragrances  [_TOPK={_TOPK}]')

In [ ]:
# Stage 5: Start FastAPI server
import threading, time, httpx, nest_asyncio, asyncio
nest_asyncio.apply()
import uvicorn
from vibescents.backend_app import create_app

_engine = globals().get('ENGINE')
if _engine is None:
    raise RuntimeError('ENGINE not defined. Run Stage 4 first.')

APP = create_app(engine=_engine)
_cfg = uvicorn.Config(app=APP, host=FASTAPI_HOST, port=FASTAPI_PORT,
                      log_level='warning', loop='asyncio', access_log=False)
_srv = uvicorn.Server(config=_cfg)

def _run(): asyncio.run(_srv.serve())
threading.Thread(target=_run, daemon=True, name='uvicorn').start()

_dl = time.time() + 30
while time.time() < _dl:
    try:
        if httpx.get(f'{_BASE_URL}/healthz', timeout=2.0).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(0.3)
else:
    raise RuntimeError('FastAPI did not start in 30s')

print(f'FastAPI ready at {_BASE_URL}')
print(f'Health: {httpx.get(f"{_BASE_URL}/healthz").json()}')

In [ ]:
# Stage 6: Dual Tunnel — Cloudflare primary + ngrok backup
# [KAGGLE] Same approach as Colab — Cloudflare binary + pyngrok both work fine.
# Pitfall: Kaggle's outbound ports are open, but some networks block port 4443.
# If Cloudflare URL never appears, use ngrok as primary.
import re, subprocess, time
CLOUDFLARE_URL = None; NGROK_URL = None

print('Starting Cloudflare Tunnel ...')
_cf = None
try:
    _cf = subprocess.Popen(
        [CLOUDFLARED_BIN, 'tunnel', '--url', f'http://localhost:{FASTAPI_PORT}', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    import atexit; atexit.register(lambda: _cf.kill() if _cf.poll() is None else None)
except Exception as _e:
    print(f'WARNING: cloudflared failed to start: {_e}')

_pat = re.compile(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com')
_dl  = time.time() + 60
while _cf is not None and time.time() < _dl:
    line = _cf.stdout.readline()
    if not line: time.sleep(0.2); continue
    m = _pat.search(line)
    if m: CLOUDFLARE_URL = m.group(0); break

if CLOUDFLARE_URL: print(f'Cloudflare: {CLOUDFLARE_URL}')
else: print('WARNING: Cloudflare URL not found within 60s')

print('Starting ngrok backup ...')
try:
    from pyngrok import ngrok as _ng
    _tun = _ng.connect(FASTAPI_PORT)
    NGROK_URL = _tun.public_url
    if NGROK_URL.startswith('http://'): NGROK_URL = 'https://' + NGROK_URL[7:]
    print(f'ngrok: {NGROK_URL}')
except Exception as _e:
    print(f'ngrok unavailable: {_e}')

ACTIVE_BACKEND_URL = CLOUDFLARE_URL or NGROK_URL
print('=' * 70)
print(f'PRIMARY (Cloudflare): {CLOUDFLARE_URL or "NOT AVAILABLE"}')
print(f'BACKUP  (ngrok):      {NGROK_URL or "NOT AVAILABLE"}')
print(f'\nSet in frontend .env.local:')
print(f'  ML_BACKEND_URL={ACTIVE_BACKEND_URL or "<paste URL>"}')
print('=' * 70)

In [ ]:
# Stage 7: Locked demo cache — 5 context-only fallback responses
import json, os, time, httpx
from tqdm.auto import tqdm

_WHITE_PNG_B64 = (
    'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNk'
    '+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg=='
)

LOCKED_CASES = [
    {'key': 'Gala|Evening|Mysterious',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Gala', 'timeOfDay': 'Evening', 'mood': 'Mysterious', 'customNotes': None}}},
    {'key': 'Business|Morning|Bold',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Business', 'timeOfDay': 'Morning', 'mood': 'Bold', 'customNotes': None}}},
    {'key': 'Date Night|Evening|Warm',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Date Night', 'timeOfDay': 'Evening', 'mood': 'Warm', 'customNotes': None}}},
    {'key': 'Casual|Afternoon|Fresh',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Casual', 'timeOfDay': 'Afternoon', 'mood': 'Fresh', 'customNotes': None}}},
    {'key': 'Wedding|Evening|Subtle',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Wedding', 'timeOfDay': 'Evening', 'mood': 'Subtle', 'customNotes': None}}},
]

local_cache_path = os.path.join(W5_ARTIFACTS, 'locked_responses.json')
os.makedirs(os.path.dirname(local_cache_path), exist_ok=True)
locked_responses = {}

_pb = tqdm(LOCKED_CASES, desc='Generating locked responses', ncols=90, unit='case')
for case in _pb:
    key = case['key']; _pb.set_postfix_str(key)
    try:
        resp = httpx.post(f'{_BASE_URL}/recommend', json=case['request'], timeout=180.0)
        resp.raise_for_status()
        locked_responses[key] = resp.json()
        _n   = len(locked_responses[key].get('recommendations', []))
        _top = locked_responses[key]['recommendations'][0]['name'] if _n else '?'
        print(f"  '{key}' -> {_n} recs | top: {_top}")
    except Exception as _e:
        print(f"  WARNING: '{key}' failed: {_e}")
_pb.close()

ENGINE._cache.update(locked_responses)
print(f'Injected {len(locked_responses)} responses into engine cache')

_payload = json.dumps(locked_responses, indent=2)
with open(local_cache_path, 'w') as _lf: _lf.write(_payload)
print(f'Saved: {local_cache_path}')

In [ ]:
# Stage 8: Frontend Integration
import os

_backend = globals().get('ACTIVE_BACKEND_URL') or globals().get('CLOUDFLARE_URL') or globals().get('NGROK_URL', '')

print('=' * 70)
print('  FRONTEND INTEGRATION')
print('=' * 70)
print()
print('Add to frontend .env.local (or Vercel env vars):')
print()
print(f'  ML_BACKEND_URL={_backend or "<paste Cloudflare or ngrok URL here>"}')
print()
print('The Next.js route (/api/recommend) already proxies to ML_BACKEND_URL.')
print('No file changes needed in frontend repo.')
print()
print('=' * 70)

_ref_path = os.path.join(W5_ARTIFACTS, 'backend_url.txt')
os.makedirs(os.path.dirname(_ref_path), exist_ok=True)
with open(_ref_path, 'w') as _uf:
    _uf.write(f'ML_BACKEND_URL={_backend}\n')
print(f'URL saved: {_ref_path}')

In [ ]:
# Stage 9: Smoke Test
import httpx

_PAYLOAD = {
    'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
    'context': {'eventType': 'Gala', 'timeOfDay': 'Evening', 'mood': 'Mysterious', 'customNotes': None},
}
_REQUIRED = {'rank', 'name', 'house', 'score', 'notes', 'reasoning', 'occasion'}


def _validate(data, url):
    recs = data.get('recommendations', [])
    assert recs, f'Empty recommendations from {url}'
    for i, r in enumerate(recs):
        miss = _REQUIRED - set(r.keys())
        assert not miss, f'rec[{i}] missing fields: {miss}'
    print(f'  PASS ({url}) — {len(recs)} recs')
    for r in recs: print(f'    #{r["rank"]} {r["name"]} | {r["house"]} | score={r["score"]:.4f}')


_ok = True
print('Test 1: Local server')
try:
    _r = httpx.post(f'{_BASE_URL}/recommend', json=_PAYLOAD, timeout=180.0)
    _r.raise_for_status(); _validate(_r.json(), _BASE_URL)
except Exception as _e: print(f'  FAIL: {_e}'); _ok = False

_cf_url = globals().get('CLOUDFLARE_URL')
if _cf_url:
    print(f'Test 2: Cloudflare')
    try:
        _r2 = httpx.post(f'{_cf_url}/recommend', json=_PAYLOAD, timeout=60.0, follow_redirects=True)
        _r2.raise_for_status(); _validate(_r2.json(), _cf_url)
    except Exception as _e2: print(f'  WARNING: {_e2}')

print(f'Health: {httpx.get(f"{_BASE_URL}/healthz").json()}')
print('All smoke tests PASSED' if _ok else 'WARNING: some tests failed')

In [ ]:
# Stage 10: Pre-Demo Runbook
import os

_D = '=' * 70
print(_D); print('  VIBESCENT WEEK 5 — QWEN3-VL — KAGGLE EDITION — PRE-DEMO RUNBOOK'); print(_D)
print('''
BEFORE PRESENTING (Kaggle-specific)
------------------------------------
1. Notebook settings: GPU = P100 or T4, Internet = ON
   (Settings gear icon -> Accelerator)

2. Secrets: Add-ons -> Secrets -> HF_TOKEN must be set
   (This is DIFFERENT from Colab sidebar secrets)

3. Optional speedup: add 'vibescent-artifacts' as input dataset
   -> skips Stage 0 re-embedding (~10 min saved)
   -> Save output as dataset after first successful full run

4. Session time budget: ~30 hrs/week free. Each session max 9-12 hrs.
   Plan: Stage 0 + full pipeline = ~35 min first run, ~20 min subsequent.

5. Run all stages top-to-bottom. Stage 0 skip is automatic if
   /kaggle/input/vibescent-artifacts/ contains the embeddings.

6. Copy PRIMARY tunnel URL to frontend .env.local

NO API KEYS AT INFERENCE — all models run locally on Kaggle GPU.
HF_TOKEN needed only for first-time model download (cached in session).
''')

print('TUNNEL URLS')
print('-' * 40)
print(f'  PRIMARY: {globals().get("CLOUDFLARE_URL", "NOT SET")}')
print(f'  BACKUP:  {globals().get("NGROK_URL",     "NOT SET")}')
print()

print('IF TUNNEL DROPS')
print('-' * 40)
print('A: Switch to ngrok backup')
print('B: Re-run Stage 6 only — new tunnel in ~30s')
print('C: Context-only fallback — 5 locked responses (no outfit image):')
for _c in globals().get('LOCKED_CASES', []):
    print(f'   * {_c["key"]}')
print()

print('KAGGLE-SPECIFIC PITFALLS (KNOW THESE)')
print('-' * 40)
print('  VRAM tight: two 4-bit 8B models + 540MB corpus = ~12-13 GB of 16 GB')
print('  If OOM during Stage 7/smoke test: reduce _TOPK from 5 to 3 in Stage 4')
print('  P100 (sm_6.0): no flash_attention_2, no bfloat16 -> auto-corrected')
print('  T4 (sm_7.5):   no flash_attention_2, sdpa used -> auto-corrected')
print('  Model re-downloads every session (~20 min) unless hf-cache dataset added')
print('  torch.compile skipped on P100/T4 — correct, no regression')
print()

print('ARCHITECTURE TALKING POINTS')
print('-' * 40)
print('  Embedding: Qwen3-VL-Embedding-8B (MMEB-V2 #1, 77.8%) — direct image embed')
print('  Reranker:  Qwen3-VL-Reranker-8B — sees outfit image during ranking')
print('  Zero API calls at inference — fully local on Kaggle GPU')
print('  Consistent vector space: same model for corpus and query')
print()

print('PIPELINE SUMMARY')
print('-' * 40)
print(f'  version    : {PIPELINE_VERSION}')
print(f'  GPU tier   : {globals().get("GPU_TIER", "unknown")}')
print(f'  attn       : {globals().get("ATTN_IMPL", "unknown")}')
print(f'  corpus     : {len(globals().get("FRAGRANCE_DF", []))} fragrances')
print(f'  emb model  : Qwen/Qwen3-VL-Embedding-8B')
print(f'  reranker   : Qwen/Qwen3-VL-Reranker-8B')
print(f'  _TOPK      : 5 (Kaggle 16GB) vs 8 (Colab A100)')
print(f'  API keys   : none at inference')
print()
print(_D); print('  BACKEND IS READY. GOOD LUCK!'); print(_D)